In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import rasterio
import joblib
from rasterio.transform import xy
from scipy.ndimage import binary_erosion, distance_transform_edt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                              recall_score, f1_score, classification_report)
from xgboost import XGBClassifier

print("Imports OK")

Imports OK


 Nettoie les zones inondées par érosion (retire les bordures bruitées SAR) et sélectionne des négatifs sûrs à plus de 300 m de toute inondation, éliminant ainsi les pixels ambigus avant échantillonnage.

In [2]:

DATA_DIR   = "data/raw"
OUT_DIR    = "data/processed"
FLOOD_PATH = os.path.join(DATA_DIR, "composite_flood_dualpol.tif")

os.makedirs(OUT_DIR, exist_ok=True)

POOL_PER_CLASS    = 50_000  
TARGET_PER_CLASS  = 20_000    
BUFFER_M          = 300
EROSION_ITER      = 3
RANDOM_SEED       = 42

with rasterio.open(FLOOD_PATH) as src:
    flood_arr       = src.read(1)
    flood_transform = src.transform
    flood_crs       = src.crs
    pixel_size_m    = abs(src.transform.a)

print(f"Raster de crue : {flood_arr.shape}  (résolution {pixel_size_m:.2f} m)")
print(f"Pixels inondés bruts : {(flood_arr == 1).sum():,}")

flood_mask  = (flood_arr == 1)
flood_clean = binary_erosion(flood_mask, iterations=EROSION_ITER)
print(f"Pixels inondés après érosion ({EROSION_ITER} iter) : {flood_clean.sum():,}")

buffer_pixels = int(BUFFER_M / pixel_size_m)
print(f"Buffer = {BUFFER_M}m = {buffer_pixels} pixels")

dist_to_flood = distance_transform_edt(~flood_mask)
safe_dry      = (flood_arr == 0) & (dist_to_flood > buffer_pixels)
print(f"Pixels secs sûrs (> {BUFFER_M}m) : {safe_dry.sum():,}")

r1, c1 = np.where(flood_clean)
r0, c0 = np.where(safe_dry)

rng = np.random.default_rng(RANDOM_SEED)
n1_pool = min(POOL_PER_CLASS, len(r1))
n0_pool = min(POOL_PER_CLASS, len(r0))

i1 = rng.choice(len(r1), size=n1_pool, replace=False)
i0 = rng.choice(len(r0), size=n0_pool, replace=False)

rows   = np.concatenate([r1[i1], r0[i0]])
cols   = np.concatenate([c1[i1], c0[i0]])
labels = np.concatenate([np.ones(n1_pool, np.int8), np.zeros(n0_pool, np.int8)])

print(f"\nPool initial : {len(labels):,} points "
      f"({n1_pool:,} inondés / {n0_pool:,} secs)")

Raster de crue : (17043, 14837)  (résolution 9.49 m)
Pixels inondés bruts : 4,436,649
Pixels inondés après érosion (3 iter) : 2,305,973
Buffer = 300m = 31 pixels
Pixels secs sûrs (> 300m) : 206,591,702

Pool initial : 100,000 points (50,000 inondés / 50,000 secs)


Échantillonne les 10 facteurs aux coordonnées du pool, supprime les lignes contenant des `NaN` (nodata résiduels) puis sélectionne exactement 20 000 points par classe pour produire un dataset d'entraînement strictement 50/50.

In [ ]:
FACTORS = ["DEM", "Slope", "Aspect", "Curvature", "TWI",
           "NDVI", "NDBI", "DD", "DR", "GFI"]

xs, ys = xy(flood_transform, rows, cols)
pts    = list(zip(xs, ys))

data = {}
for name in FACTORS:
    path = os.path.join(DATA_DIR, f"{name}.tif")
    with rasterio.open(path) as src:
        nodata = src.nodata
        vals   = np.array([v[0] for v in src.sample(pts)], dtype=np.float32)
        if nodata is not None:
            vals[vals == nodata] = np.nan
        vals[~np.isfinite(vals)] = np.nan
        data[name] = vals
    print(f"  {name:10s} : {np.isnan(vals).sum():>6,} NaN / {len(vals):,}")

pool_df = pd.DataFrame(data)
pool_df.insert(0, "label", labels)

n_before = len(pool_df)
pool_df  = pool_df.dropna().reset_index(drop=True)
n_after  = len(pool_df)
print(f"\nNettoyage : {n_before:,} → {n_after:,} lignes "
      f"(supprimé {n_before - n_after:,} NaN)")

counts = pool_df["label"].value_counts().sort_index()
print(f"\nAprès nettoyage :")
print(f"  classe 0 (sec)    : {counts.get(0, 0):,}")
print(f"  classe 1 (inondé) : {counts.get(1, 0):,}")

n_final = min(TARGET_PER_CLASS, counts.get(0, 0), counts.get(1, 0))
print(f"\nObjectif : {n_final:,} par classe (50/50 strict)")

sampled = []
for cls in [0, 1]:
    subset = pool_df[pool_df["label"] == cls]
    sampled.append(subset.sample(n=n_final, random_state=RANDOM_SEED))

df = (pd.concat(sampled, ignore_index=True)
        .sample(frac=1, random_state=RANDOM_SEED)
        .reset_index(drop=True))

print(f"\nÉchantillon ÉQUILIBRÉ : {len(df):,} lignes")
print(df["label"].value_counts().sort_index())
print(f"\nShape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")
df.head()

  DEM        :      0 NaN / 100,000
  Slope      :      0 NaN / 100,000
  Aspect     :      0 NaN / 100,000
  Curvature  : 27,749 NaN / 100,000
  TWI        :      0 NaN / 100,000
  NDVI       : 27,743 NaN / 100,000
  NDBI       : 27,743 NaN / 100,000
  DD         :      0 NaN / 100,000
  DR         :      0 NaN / 100,000
  GFI        :      0 NaN / 100,000

Nettoyage : 100,000 → 72,251 lignes (supprimé 27,749 NaN)

Après nettoyage :
  classe 0 (sec)    : 22,251
  classe 1 (inondé) : 50,000

Objectif : 20,000 par classe (50/50 strict)

Échantillon ÉQUILIBRÉ : 40,000 lignes
label
0    20000
1    20000
Name: count, dtype: int64

Shape : (40000, 11)
Colonnes : ['label', 'DEM', 'Slope', 'Aspect', 'Curvature', 'TWI', 'NDVI', 'NDBI', 'DD', 'DR', 'GFI']


,label,DEM,Slope,Aspect,Curvature,TWI,NDVI,NDBI,DD,DR,GFI
0,1,42.0,0.000000,-1.0,-0.000000,11.673245,0.097009,0.136748,0.644075,562.133301,0.500000
1,0,39.0,0.000000,-1.0,-0.000000,10.286951,0.149226,0.100045,0.619876,1777.621582,0.083333
2,1,26.0,2.430954,45.0,-0.232219,12.484176,0.312047,-0.004184,1.273557,1124.266602,0.142857
3,0,49.0,0.000000,-1.0,-0.000000,12.851901,0.251629,0.072995,0.687731,562.133301,0.250000
4,1,33.0,0.000000,-1.0,-0.000000,13.120165,0.240253,0.026006,1.239404,562.133301,0.714286


Crée 5 nouvelles variables dérivées (ratios et indices composites) à partir des 10 facteurs bruts, afin de capturer des interactions topographiques et hydrologiques que le modèle seul ne pourrait pas découvrir.

In [4]:
df_fe = df.copy()

df_fe["DEM_Slope"]    = df_fe["DEM"] / (df_fe["Slope"] + 1)
df_fe["TWI_DD"]       = df_fe["TWI"] * df_fe["DD"]
df_fe["NDVI_NDBI"]    = df_fe["NDVI"] - df_fe["NDBI"]
df_fe["Vulnerability"] = (
    -df_fe["DEM"].rank(pct=True)
    + df_fe["TWI"].rank(pct=True)
    + df_fe["DD"].rank(pct=True)
    - df_fe["NDVI"].rank(pct=True)
)
df_fe["Roughness"] = 0.0

print(f"Avant FE : {df.shape}")
print(f"Après FE : {df_fe.shape}")
print(f"\nDistribution finale : {df_fe['label'].value_counts().to_dict()}")

Avant FE : (40000, 11)
Après FE : (40000, 16)

Distribution finale : {1: 20000, 0: 20000}


Fige les distributions des 4 facteurs de `Vulnerability` pour les prédictions futures.

In [5]:
rank_dists = {
    "DEM":  np.sort(df_fe["DEM"].values),
    "TWI":  np.sort(df_fe["TWI"].values),
    "DD":   np.sort(df_fe["DD"].values),
    "NDVI": np.sort(df_fe["NDVI"].values),
}
joblib.dump(rank_dists, os.path.join(OUT_DIR, "rank_distributions.pkl"))
print(" rank_distributions.pkl sauvegardé (sur données ÉQUILIBRÉES)")
for k, v in rank_dists.items():
    print(f"   {k:5s} : [{v.min():.2f}, {v.max():.2f}]  n={len(v):,}")

 rank_distributions.pkl sauvegardé (sur données ÉQUILIBRÉES)
   DEM   : [0.00, 103.00]  n=40,000
   TWI   : [0.00, 26.43]  n=40,000
   DD    : [0.00, 2.00]  n=40,000
   NDVI  : [-0.10, 0.64]  n=40,000


Sépare le dataset en 70 % train / 30 % test (stratifé), applique un `StandardScaler` ajusté uniquement sur le train, puis entraîne trois modèles de base (RandomForest, SVM, XGBoost) et calcule leurs métriques sur le test.

In [7]:
FEATURES = [c for c in df_fe.columns if c != "label"]
X = df_fe[FEATURES]
y = df_fe["label"]

print(f"Features ({len(FEATURES)}) : {FEATURES}")
print(f"Distribution X : {y.value_counts().to_dict()}\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train : {X_train.shape}  →  {y_train.value_counts().to_dict()}")
print(f"Test  : {X_test.shape}   →  {y_test.value_counts().to_dict()}")

rf  = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced")
svm = SVC(probability=True, random_state=42, class_weight="balanced")
xgb = XGBClassifier(n_estimators=200, random_state=42,
                    use_label_encoder=False, eval_metric="logloss",
                    n_jobs=-1, verbosity=0)

models = {"RandomForest": rf, "SVM": svm, "XGBoost": xgb}
results = {}

for name, model in models.items():
    print(f"\n Entraînement {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        "Accuracy":  accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall":    recall_score(y_test, y_pred),
        "F1":        f1_score(y_test, y_pred),
        "AUC-ROC":   roc_auc_score(y_test, y_proba),
    }
    print(f"   AUC = {results[name]['AUC-ROC']:.4f}")

Features (15) : ['DEM', 'Slope', 'Aspect', 'Curvature', 'TWI', 'NDVI', 'NDBI', 'DD', 'DR', 'GFI', 'DEM_Slope', 'TWI_DD', 'NDVI_NDBI', 'Vulnerability', 'Roughness']
Distribution X : {1: 20000, 0: 20000}

Train : (28000, 15)  →  {1: 14000, 0: 14000}
Test  : (12000, 15)   →  {1: 6000, 0: 6000}

 Entraînement RandomForest...
   AUC = 0.9791

 Entraînement SVM...
   AUC = 0.9474

 Entraînement XGBoost...
   AUC = 0.9845


Combine RandomForest, SVM et XGBoost via un méta-modèle LogisticRegression (validation croisée 5-fold), puis affiche les métriques détaillées : matrice de confusion, TP/TN/FP/FN, sensibilité, spécificité et taux d'erreur.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)

if "results" not in dir() or not isinstance(results, dict):
    results = {}

# --- Entraînement ---
print("Entraînement du Stacking...\n")

stacking = StackingClassifier(
    estimators=[
        ("rf",  RandomForestClassifier(n_estimators=200, random_state=42,
                                       n_jobs=-1, class_weight="balanced")),
        ("svm", SVC(probability=True, random_state=42, class_weight="balanced")),
        ("xgb", XGBClassifier(n_estimators=200, random_state=42,
                              use_label_encoder=False, eval_metric="logloss",
                              n_jobs=-1, verbosity=0)),
    ],
    final_estimator=LogisticRegression(random_state=42, max_iter=1000),
    cv=5, n_jobs=-1, passthrough=False,
)

stacking.fit(X_train_scaled, y_train)
y_pred  = stacking.predict(X_test_scaled)
y_proba = stacking.predict_proba(X_test_scaled)[:, 1]

results["Stacking"] = {
    "Accuracy":  accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall":    recall_score(y_test, y_pred),
    "F1":        f1_score(y_test, y_pred),
    "AUC-ROC":   roc_auc_score(y_test, y_proba),
}

cm = confusion_matrix(y_test, y_pred)

print("=" * 70)
print("VÉRIFICATION DE LA MATRICE")
print("=" * 70)
print(f"  Shape  : {cm.shape}")
print(f"  Labels uniques dans y_test : {sorted(set(y_test))}")
print(f"  Labels uniques dans y_pred : {sorted(set(y_pred))}")
print(f"  Matrice :\n{cm}\n")

if cm.shape != (2, 2):
    print("=" * 70)
    print("ERREUR : la matrice n'est pas 2x2")
    print("=" * 70)
    print("Causes possibles :")
    print("  - y_test contient autre chose que 0 et 1")
    print("  - le modèle prédit des classes supplémentaires")
    print("  - relance le split train/test au début du notebook")
    raise ValueError(f"Matrice de shape {cm.shape} au lieu de (2, 2)")

tn, fp, fn, tp = cm.ravel()
total = tn + fp + fn + tp

print("=" * 70)
print("STACKING — RÉSULTATS SUR LE TEST SET (30%)")
print("=" * 70)

m = results["Stacking"]
print(f"  Accuracy  (Exactitude)         : {m['Accuracy']:.4f}  ({m['Accuracy']*100:.2f}%)")
print(f"  Precision (Précision)          : {m['Precision']:.4f}  ({m['Precision']*100:.2f}%)")
print(f"  Recall    (Rappel/Sensibilité) : {m['Recall']:.4f}  ({m['Recall']*100:.2f}%)")
print(f"  F1-score  (Compromis P/R)      : {m['F1']:.4f}  ({m['F1']*100:.2f}%)")
print(f"  AUC-ROC   (Discrimination)     : {m['AUC-ROC']:.4f}")

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT (par classe)")
print("=" * 70)
print(classification_report(
    y_test, y_pred,
    target_names=["Sec (0)", "Inondé (1)"],
    digits=4,
))

print("=" * 70)
print("MATRICE DE CONFUSION")
print("=" * 70)
print(f"\n                      Prédit SEC    Prédit INONDÉ")
print(f"  Réel SEC            {tn:>10,}    {fp:>12,}")
print(f"  Réel INONDÉ         {fn:>10,}    {tp:>12,}")

print("\n" + "=" * 70)
print("DÉTAIL DES 4 CAS")
print("=" * 70)
print(f"  Vrais Négatifs  (TN)  sec     -> sec      : {tn:>10,}   ({100*tn/total:.2f}% du total)")
print(f"  Vrais Positifs  (TP)  inondé  -> inondé   : {tp:>10,}   ({100*tp/total:.2f}% du total)")
print(f"  Faux Positifs   (FP)  sec     -> inondé   : {fp:>10,}   ({100*fp/total:.2f}% du total)")
print(f"  Faux Négatifs   (FN)  inondé  -> sec      : {fn:>10,}   ({100*fn/total:.2f}% du total)   <- le plus grave en risque")

print("\n" + "=" * 70)
print("MÉTRIQUES DÉRIVÉES")
print("=" * 70)

specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"  Sensibilité (Recall / TPR)     : {100*m['Recall']:.2f}%   — le modèle détecte {100*m['Recall']:.1f}% des vraies inondations")
print(f"  Spécificité (TNR)              : {100*specificity:.2f}%   — le modèle reconnaît {100*specificity:.1f}% des zones sèches")
print(f"  Précision (PPV)                : {100*m['Precision']:.2f}%   — {100*m['Precision']:.1f}% des alertes sont correctes")
print(f"  Valeur Prédictive Négative     : {100*npv:.2f}%   — {100*npv:.1f}% des prédictions 'sec' sont correctes")
print(f"  Taux de Faux Positifs  (FPR)   : {100*fpr:.2f}%   — {100*fpr:.1f}% de fausses alertes")
print(f"  Taux de Faux Négatifs  (FNR)   : {100*fnr:.2f}%   — {100*fnr:.1f}% d'inondations ratées")

print("\n" + "=" * 70)
print("RÉSUMÉ EN UNE LIGNE")
print("=" * 70)
print(f"  Sur {total:,} prédictions : {tp+tn:,} correctes ({100*(tp+tn)/total:.1f}%) · {fp+fn:,} erreurs ({100*(fp+fn)/total:.1f}%)")

Entraînement du Stacking...

VÉRIFICATION DE LA MATRICE
  Shape  : (2, 2)
  Labels uniques dans y_test : [0, 1]
  Labels uniques dans y_pred : [np.int8(0), np.int8(1)]
  Matrice :
[[5666  334]
 [ 318 5682]]

STACKING — RÉSULTATS SUR LE TEST SET (30%)
  Accuracy  (Exactitude)         : 0.9457  (94.57%)
  Precision (Précision)          : 0.9445  (94.45%)
  Recall    (Rappel/Sensibilité) : 0.9470  (94.70%)
  F1-score  (Compromis P/R)      : 0.9457  (94.57%)
  AUC-ROC   (Discrimination)     : 0.9845

CLASSIFICATION REPORT (par classe)
              precision    recall  f1-score   support

     Sec (0)     0.9469    0.9443    0.9456      6000
  Inondé (1)     0.9445    0.9470    0.9457      6000

    accuracy                         0.9457     12000
   macro avg     0.9457    0.9457    0.9457     12000
weighted avg     0.9457    0.9457    0.9457     12000

MATRICE DE CONFUSION

                      Prédit SEC    Prédit INONDÉ
  Réel SEC                 5,666             334
  Réel INONDÉ  

**Sauvegarde du modèle** — Export `.pkl` du Stacking, scaler et features pour Streamlit.

In [ ]:
joblib.dump(stacking,   os.path.join(OUT_DIR, "stacking_model_improvedV2.pkl"))
joblib.dump(scaler,     os.path.join(OUT_DIR, "scaler_improvedV2.pkl"))
joblib.dump(FEATURES,   os.path.join(OUT_DIR, "feature_names_improvedV2.pkl"))

print("=" * 60)
print("FICHIERS SAUVEGARDÉS POUR STREAMLIT")
print("=" * 60)
for fname in ["stacking_model_improvedV2.pkl", "scaler_improvedV2.pkl",
              "feature_names_improvedV2.pkl", "rank_distributions.pkl"]:
    p = os.path.join(OUT_DIR, fname)
    size = os.path.getsize(p) / 1e6 if os.path.exists(p) else 0
    status = "✅" if os.path.exists(p) else "❌"
    print(f"  {status} {fname:35s} {size:6.2f} MB")

print(f"\nDistribution d'entraînement : {y.value_counts().to_dict()}")
print(f"Features ({len(FEATURES)}) : {FEATURES}")

FICHIERS SAUVEGARDÉS POUR STREAMLIT
  ✅ stacking_model_improvedV2.pkl        71.80 MB
  ✅ scaler_improvedV2.pkl                 0.00 MB
  ✅ feature_names_improvedV2.pkl          0.00 MB
  ✅ rank_distributions.pkl                0.64 MB

Distribution d'entraînement : {1: 20000, 0: 20000}
Features (15) : ['DEM', 'Slope', 'Aspect', 'Curvature', 'TWI', 'NDVI', 'NDBI', 'DD', 'DR', 'GFI', 'DEM_Slope', 'TWI_DD', 'NDVI_NDBI', 'Vulnerability', 'Roughness']
